In [1]:
print("Ok")

Ok


In [2]:
%pwd

'c:\\Users\\youse\\Downloads\\End-to-end-Medical-Chatbot-Generative-AI-main\\End-to-end-Medical-Chatbot-Generative-AI-main\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\youse\\Downloads\\End-to-end-Medical-Chatbot-Generative-AI-main\\End-to-end-Medical-Chatbot-Generative-AI-main'

In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\conda\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
#Extract Data From the PDF File
def load_pdf_file(data):
    """
    Summary:
        Read every PDF file inside the given folder and return the parsed documents.

    Args:
        data (str): Folder path that contains the PDF files to load.

    Returns/Output:
        list: A list of Document objects built from the PDF content.

    NextStep:
        Send the returned documents to text_split() so they can be broken into smaller chunks.
    """
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents


In [ ]:
extracted_data = load_pdf_file(data="Data/")

In [8]:
# extracted_data

In [35]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    """
    Summary:
        Break the loaded documents into smaller chunks so they are easier to search and embed.

    Args:
        extracted_data (list): The list of Document objects returned by load_pdf_file().

    Returns/Output:
        list: A list of smaller Document chunks ready for embedding.

    NextStep:
        Pass these chunks to download_hugging_face_embeddings() and then store them in Pinecone.
    """
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 7020


In [11]:
# text_chunks

In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    """
    Summary:
        Load the sentence-transformer model that creates embeddings for the medical chatbot.

    Args:
        None

    Returns/Output:
        HuggingFaceEmbeddings: The embedding model used to turn text into vectors.

    NextStep:
        Use the returned embedding model with PineconeVectorStore.from_documents() to build the index.
    """
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [14]:
embeddings = download_hugging_face_embeddings()

C:\Users\youse\AppData\Local\Temp\ipykernel_45336\1196424635.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3303.59it/s]


In [15]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [16]:
# query_result

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [19]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(
    api_key=PINECONE_API_KEY  # Use environment variable instead of hardcoded key
)

index_name = "medicalbot"

if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Pinecone ready ")

Pinecone ready 


In [20]:
from langchain_pinecone import Pinecone as PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name="medicalbot",
    embedding=embeddings
)

retriever = docsearch.as_retriever()

In [21]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [22]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [23]:
docsearch

In [24]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [25]:
retrieved_docs = retriever.invoke("What is Acne?")

In [26]:
retrieved_docs

[Document(id='5cf118de-7748-4225-97d7-c119e245d868', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 37.0, 'page_label': '38', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='Acidosis seeRespiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when thepores of the skin become clogged with oil, dead skincells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is'),
 Document(id='d055df6f-9c15-4804-bb51-4cd00e530734', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 37.0, 'page_label': '38', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='Acidosis seeRespiratory aci

In [27]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [31]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print("Answer:", response["answer"])

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [32]:
try:
    response = rag_chain.invoke({"input": "What is stats?"})
    print("Answer:", response["answer"])
except Exception as e:
    print(f"Error: {type(e).__name__}: {str(e)}")
    print("\nNote: If you see an OpenAI quota error, verify your API key and billing.")
    print("If you see other errors, the RAG chain is now working correctly!")

Error: RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Note: If you see an OpenAI quota error, verify your API key and billing.
If you see other errors, the RAG chain is now working correctly!


In [33]:
# Verify API keys are loaded from .env
print("API Keys Status:")
print(f"PINECONE_API_KEY loaded: {'Yes' if PINECONE_API_KEY else 'No'}")
print(f"OPENAI_API_KEY loaded: {'Yes' if OPENAI_API_KEY else 'No'}")
print(f"\nPINECONE key starts with: {PINECONE_API_KEY[:10]}...")
print(f"OPENAI key starts with: {OPENAI_API_KEY[:15]}...")
print("\nBoth API keys are now loaded from .env successfully!")

API Keys Status:
PINECONE_API_KEY loaded: Yes
OPENAI_API_KEY loaded: Yes

PINECONE key starts with: pcsk_5it1j...
OPENAI key starts with: sk-proj-EfwJvPO...

Both API keys are now loaded from .env successfully!
